<a href="https://colab.research.google.com/github/kanchanr8595-create/CareFlow-Clinical-Pathway-Process-Mining/blob/development_akshaya/summary_carfelow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import numpy as np
import pandas as pd

df = pd.read_csv(r"/content/careflow_synthetic_500_patients.csv")

print(df.isnull().sum())

print("Total missing values:", df.isnull().sum().sum())

missing_rows = df[df.isnull().any(axis=1)]
print(missing_rows)

print((df.isnull().sum() / len(df) * 100).round(2))

df["event_order"] = pd.to_numeric(
    df["event_order"],
    errors="coerce"
)

df_sorted = df.sort_values(
    ["case_id", "event_order"]
).copy()

df_sorted["previous_timestamp"] = (
    df_sorted.groupby("case_id")["timestamp"]
    .shift(1)
)

backward_timestamps = df_sorted[
    df_sorted["timestamp"]
    < df_sorted["previous_timestamp"]
]

print("Backward timestamp sequences:")
print(backward_timestamps[
    [
        "case_id",
        "event_order",
        "activity",
        "previous_timestamp",
        "timestamp"
    ]
])

print(
    "\nNumber of backward timestamp sequences:",
    len(backward_timestamps)
)

df_sorted = df.sort_values(
    ["case_id", "event_order"]
).copy()

df_sorted["previous_activity"] = (
    df_sorted.groupby("case_id")["activity"].shift(1)
)

repeated_activities = df_sorted[
    df_sorted["activity"]
    == df_sorted["previous_activity"]
]

print("Repeated consecutive activities:")
print(
    repeated_activities[
        [
            "case_id",
            "event_order",
            "previous_activity",
            "activity",
            "timestamp"
        ]
    ]
)

print(
    "\nNumber of repeated consecutive activities:",
    len(repeated_activities)
)

df["timestamp"]=pd.to_datetime(df["timestamp"],errors="coerce")
df= df.sort_values(
    ["case_id", "timestamp", "event_order"]
).copy()

df["next_activity"] = (
    df.groupby("case_id")["activity"].shift(-1)
)

df["next_timestamp"] = (
    df.groupby("case_id")["timestamp"].shift(-1)
)

df["transition_minutes"] = (
    df["next_timestamp"]
    - df["timestamp"]
).dt.total_seconds() / 60

print(
    df[
        [
            "case_id",
            "activity",
            "timestamp",
            "next_activity",
            "next_timestamp",
            "transition_minutes"
        ]
    ].head(20)
)

negative_times = df_sorted[
    df["transition_minutes"] < 0
]

print("\nNegative transition times:", len(negative_times))

long_gaps = df[
    df["transition_minutes"] > 180
]

print("\nTransitions exceeding 180 minutes:")
print(long_gaps[
    [
        "case_id",
        "activity",
        "next_activity",
        "transition_minutes"
    ]
])
print("\n Number of unique cases requiring reviews:",long_gaps["case_id"].nunique())



summary = {
    "Total records": len(df),

    "Unique patient cases": df["case_id"].nunique(),

    "Total missing values": df.isnull().sum(),

    "Exact duplicate rows": df.duplicated().sum(),

    "Invalid timestamps": df["timestamp"].isna().sum(),

    "Backward timestamp sequences": len(backward_timestamps),

    "Negative transition times": len(negative_times),

    "Transitions over 180 minutes": len(long_gaps),

    "Repeated consecutive activities": len(repeated_activities)


}

print("CAREFLOW DATA QUALITY REPORT")
print("-" * 40)

for check, result in summary.items():
    print(f"{check}: {result}")



    summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["Data Quality Check", "Result"]
)

summary_df.to_csv(
    "careflow_data_quality_summary.csv",
    index=False
)

print("Data-quality summary saved successfully.")


case_id         0
event_id        0
event_order     0
activity        0
timestamp       0
department      0
journey_type    0
is_loop_back    0
data_source     0
dtype: int64
Total missing values: 0
Empty DataFrame
Columns: [case_id, event_id, event_order, activity, timestamp, department, journey_type, is_loop_back, data_source]
Index: []
case_id         0.0
event_id        0.0
event_order     0.0
activity        0.0
timestamp       0.0
department      0.0
journey_type    0.0
is_loop_back    0.0
data_source     0.0
dtype: float64
Backward timestamp sequences:
Empty DataFrame
Columns: [case_id, event_order, activity, previous_timestamp, timestamp]
Index: []

Number of backward timestamp sequences: 0
Repeated consecutive activities:
Empty DataFrame
Columns: [case_id, event_order, previous_activity, activity, timestamp]
Index: []

Number of repeated consecutive activities: 0
   case_id             activity           timestamp        next_activity  \
0    P0001         Registration 2026-09